# Notebook 3: Evaluation & Iteration

## Why Evaluation Matters

An AI function without evaluation is just a guess. Evaluation gives you:
- **Baseline metrics** — how well does the current version actually perform?
- **Field-level diagnostics** — which outputs are accurate and which need work?
- **Iteration evidence** — quantified proof that V2 is better than V1

## Probabilistic Nature of LLMs

LLMs are inherently non-deterministic. Even with temperature=0 and structured output constraints:
- Accuracy scores will fluctuate +/- 5-10% between runs on small datasets
- A "100% accuracy" on 12 test rows does NOT guarantee production performance
- Focus on **directional improvement** (V1 to V2), not absolute scores
- For production confidence, use 50-100+ labeled examples

---

In [ ]:
# Prerequisite check: Ensure Notebook 2 (Function Creation) has been run
from snowflake.snowpark.context import get_active_session
session = get_active_session()
try:
    state = session.sql("SELECT NOTEBOOK_ID FROM AI_STUDIO_DEMO.PUBLIC.DEMO_STATE WHERE NOTEBOOK_ID = 'function_creation'").collect()
    if not state:
        raise Exception('Not found')
    print('Prerequisite check passed: Notebook 2 (Function Creation) has been completed.')
except:
    print('WARNING: Please run Notebook 2 (AI Function Creation) first!')
    print('   That notebook creates the CLASSIFY_SUPPORT_TICKET and V2 functions required here.')

In [ ]:
USE DATABASE AI_STUDIO_DEMO;
USE SCHEMA PUBLIC;

## Step 1: Create Labeled Test Data

Evaluation requires a table with:
- **Input column(s):** the ticket text fed to the function
- **Label column:** the expected/ground-truth output (human-verified classifications)

For multi-output functions like ours (RETURNS VARIANT), the label column is a VARIANT containing the expected JSON.

In [ ]:
CREATE OR REPLACE TABLE EVALUATION_TEST_DATA (
    TEST_ID INT,
    TICKET_TEXT VARCHAR,
    EXPECTED_OUTPUT VARIANT
);

In [ ]:
-- 12 human-labeled ground truth examples
INSERT INTO EVALUATION_TEST_DATA
SELECT 1, 'Subject: URGENT - XT-400 bond failure on automotive trim assembly\n\nOur Tier 1 line in Monterrey has been down since 6am. The XT-400 tape is delaminating. This is blocking 2,200 units/day.',
  PARSE_JSON('{"division":"Industrial Adhesives & Tapes","issue_type":"Product Defect / Failure","priority":"P1-Critical","customer_segment":"OEM","sla_flag":true}')
UNION ALL SELECT 2, 'Subject: CRITICAL - Respirator filter failure during chemical spill response\n\nDuring a chlorine gas response, workers reported breakthrough odor through cartridges. OSHA notified. Lives at stake.',
  PARSE_JSON('{"division":"Safety & Industrial","issue_type":"Product Defect / Failure","priority":"P1-Critical","customer_segment":"End User","sla_flag":true}')
UNION ALL SELECT 3, 'Subject: Production halt - thermal interface material failure\n\nAustin fab stopped production. TC-5026 thermal compound showing 3x spec resistance. $2.1M in WIP scrapped. $45M contract at risk.',
  PARSE_JSON('{"division":"Electronics & Energy","issue_type":"Product Defect / Failure","priority":"P1-Critical","customer_segment":"OEM","sla_flag":true}')
UNION ALL SELECT 4, 'Subject: MIL-SPEC compliance issue on reflective tape\n\nRetro-reflective sheeting does not meet MIL-PRF-680. DoD deadline in 3 weeks. Contract $1.2M, liquidated damages in 21 days.',
  PARSE_JSON('{"division":"Safety & Industrial","issue_type":"Regulatory / Compliance","priority":"P2-High","customer_segment":"Government/Defense","sla_flag":true}')
UNION ALL SELECT 5, 'Subject: Surgical tape adhesion failure across 3 facilities\n\n12 incidents of SecureSkin Plus tape lifting. Chief Medical Officer involved. Pulling lot from 47 facilities. FDA MedWatch filing.',
  PARSE_JSON('{"division":"Healthcare","issue_type":"Product Defect / Failure","priority":"P2-High","customer_segment":"End User","sla_flag":true}')
UNION ALL SELECT 6, 'Subject: Technical data sheet request - structural adhesives\n\nEvaluating adhesives for carbon fiber bicycle frame. Need TDS for Scotch-Weld AF163-2, DP460NS. Design phase, prototyping Q3.',
  PARSE_JSON('{"division":"Industrial Adhesives & Tapes","issue_type":"Technical Specification Inquiry","priority":"P3-Standard","customer_segment":"OEM","sla_flag":false}')
UNION ALL SELECT 7, 'Subject: Requesting samples of cold-shrink splice kits\n\nPlanning to replace 45 splice joints on 15kV system. Need 3 samples of QS-II series. Project starts June 15th.',
  PARSE_JSON('{"division":"Electronics & Energy","issue_type":"Sample Request","priority":"P3-Standard","customer_segment":"End User","sla_flag":false}')
UNION ALL SELECT 8, 'Subject: Standard RMA for defective transfer tape\n\n4 of 20 rolls have contamination. Requesting RMA. Not urgent - sufficient inventory.',
  PARSE_JSON('{"division":"Industrial Adhesives & Tapes","issue_type":"Warranty / RMA","priority":"P3-Standard","customer_segment":"Converter","sla_flag":false}')
UNION ALL SELECT 9, 'Subject: 2026 product catalog and pricing request\n\nNeed updated catalog and distributor price list. No rush - 2-3 weeks is fine.',
  PARSE_JSON('{"division":"Industrial Adhesives & Tapes","issue_type":"Technical Specification Inquiry","priority":"P4-Low","customer_segment":"Distributor","sla_flag":false}')
UNION ALL SELECT 10, 'Subject: Future project scoping - adhesive for cobot joints\n\nEarly R&D, 18-month timeline. Bonding aluminum to PEEK. Just starting a conversation.',
  PARSE_JSON('{"division":"Industrial Adhesives & Tapes","issue_type":"Application Engineering Support","priority":"P4-Low","customer_segment":"OEM","sla_flag":false}')
UNION ALL SELECT 11, 'Subject: Pricing dispute on last invoice\n\nInvoice shows $347/case for Command strips but contract says $312/case. Credit $2,940 difference.',
  PARSE_JSON('{"division":"Consumer","issue_type":"Pricing / Contract Dispute","priority":"P3-Standard","customer_segment":"End User","sla_flag":false}')
UNION ALL SELECT 12, 'Subject: REACH compliance declaration needed for 5 materials\n\nEU audit March 28. Missing REACH declarations for TC-5026, Novec 7100, VHB 5952. Need within 72 hours. $200M shipments at risk.',
  PARSE_JSON('{"division":"Electronics & Energy","issue_type":"Regulatory / Compliance","priority":"P2-High","customer_segment":"OEM","sla_flag":true}');

## Step 2: Evaluate V1 (Baseline)

We measure field-level accuracy by comparing function output to ground truth with exact string matching.

In [ ]:
-- V1 field-level accuracy
SELECT
  'V1 (baseline)' AS version,
  COUNT(*) AS total_tests,
  ROUND(AVG(IFF(EXPECTED_OUTPUT:priority::VARCHAR = c:priority::VARCHAR, 1.0, 0.0)), 3) AS priority_acc,
  ROUND(AVG(IFF(EXPECTED_OUTPUT:division::VARCHAR = c:division::VARCHAR, 1.0, 0.0)), 3) AS division_acc,
  ROUND(AVG(IFF(EXPECTED_OUTPUT:issue_type::VARCHAR = c:issue_type::VARCHAR, 1.0, 0.0)), 3) AS issue_type_acc,
  ROUND(AVG(IFF(EXPECTED_OUTPUT:customer_segment::VARCHAR = c:customer_segment::VARCHAR, 1.0, 0.0)), 3) AS segment_acc,
  ROUND(AVG(IFF(EXPECTED_OUTPUT:sla_flag::BOOLEAN = c:sla_flag::BOOLEAN, 1.0, 0.0)), 3) AS sla_flag_acc
FROM EVALUATION_TEST_DATA t,
LATERAL (SELECT CLASSIFY_SUPPORT_TICKET(t.TICKET_TEXT) AS c);

### V1 Findings

Expected approximate results (will vary by run):
- **division: ~100%** — model nails this
- **sla_flag: ~100%** — binary decision, easy for the model
- **priority: ~75%** — some borderline P2/P3 disagreements
- **issue_type: ~8%** — critical failure! Model paraphrases instead of selecting from enum

The issue_type failure is the key insight: **prompt instructions are soft guidance, not hard constraints.**

---

## Step 3: Evaluate V2 (Improved)

In [ ]:
-- V2 field-level accuracy
SELECT
  'V2 (improved)' AS version,
  COUNT(*) AS total_tests,
  ROUND(AVG(IFF(EXPECTED_OUTPUT:priority::VARCHAR = c:priority::VARCHAR, 1.0, 0.0)), 3) AS priority_acc,
  ROUND(AVG(IFF(EXPECTED_OUTPUT:division::VARCHAR = c:division::VARCHAR, 1.0, 0.0)), 3) AS division_acc,
  ROUND(AVG(IFF(EXPECTED_OUTPUT:issue_type::VARCHAR = c:issue_type::VARCHAR, 1.0, 0.0)), 3) AS issue_type_acc,
  ROUND(AVG(IFF(EXPECTED_OUTPUT:customer_segment::VARCHAR = c:customer_segment::VARCHAR, 1.0, 0.0)), 3) AS segment_acc,
  ROUND(AVG(IFF(EXPECTED_OUTPUT:sla_flag::BOOLEAN = c:sla_flag::BOOLEAN, 1.0, 0.0)), 3) AS sla_flag_acc
FROM EVALUATION_TEST_DATA t,
LATERAL (SELECT CLASSIFY_SUPPORT_TICKET_V2(t.TICKET_TEXT) AS c);

### V2 Improvement

Expected result: **issue_type jumps from ~8% to ~90%+** thanks to enum constraints in the response_format schema. Other fields remain stable.

Key takeaway: `response_format` enum constraints provide **token-level enforcement** that prompt instructions alone cannot guarantee.

---

## Step 4: Custom Evaluation Metric

For multi-output functions, you often want **weighted scoring** across fields. Priority matters more than product_family, for example.

In [ ]:
-- Custom metric: weighted field accuracy
-- Priority 40%, Division 30%, Issue Type 20%, SLA Flag 10%
CREATE OR REPLACE FUNCTION WEIGHTED_FIELD_ACCURACY(
    EXPECTED VARCHAR, 
    PREDICTED VARCHAR
)
RETURNS OBJECT
LANGUAGE SQL
AS
$$
SELECT OBJECT_CONSTRUCT(
  'score',
    (CASE WHEN PARSE_JSON(EXPECTED):priority::VARCHAR = PARSE_JSON(PREDICTED):priority::VARCHAR THEN 0.4 ELSE 0.0 END) +
    (CASE WHEN PARSE_JSON(EXPECTED):division::VARCHAR = PARSE_JSON(PREDICTED):division::VARCHAR THEN 0.3 ELSE 0.0 END) +
    (CASE WHEN PARSE_JSON(EXPECTED):issue_type::VARCHAR = PARSE_JSON(PREDICTED):issue_type::VARCHAR THEN 0.2 ELSE 0.0 END) +
    (CASE WHEN PARSE_JSON(EXPECTED):sla_flag::BOOLEAN = PARSE_JSON(PREDICTED):sla_flag::BOOLEAN THEN 0.1 ELSE 0.0 END),
  'feedback',
    CONCAT(
      'Priority: ', IFF(PARSE_JSON(EXPECTED):priority::VARCHAR = PARSE_JSON(PREDICTED):priority::VARCHAR, 'PASS', 'FAIL'),
      ' | Division: ', IFF(PARSE_JSON(EXPECTED):division::VARCHAR = PARSE_JSON(PREDICTED):division::VARCHAR, 'PASS', 'FAIL'),
      ' | Issue Type: ', IFF(PARSE_JSON(EXPECTED):issue_type::VARCHAR = PARSE_JSON(PREDICTED):issue_type::VARCHAR, 'PASS', 'FAIL'),
      ' | SLA Flag: ', IFF(PARSE_JSON(EXPECTED):sla_flag::BOOLEAN = PARSE_JSON(PREDICTED):sla_flag::BOOLEAN, 'PASS', 'FAIL')
    )
)
$$;

In [ ]:
-- Test the custom metric
SELECT 
  r['score']::FLOAT AS score,
  r['feedback']::VARCHAR AS feedback
FROM (
  SELECT WEIGHTED_FIELD_ACCURACY(
    '{"division":"Safety & Industrial","priority":"P1-Critical","issue_type":"Product Defect / Failure","sla_flag":true}',
    '{"division":"Safety & Industrial","priority":"P2-High","issue_type":"Product Defect / Failure","sla_flag":false}'
  ) AS r
);

The custom metric returns 0.5 for the test above: Division PASS (0.3) + Issue Type PASS (0.2) = 0.5, but Priority FAIL (missed 0.4) and SLA Flag FAIL (missed 0.1).

---

## Step 5: Formal Evaluation via Stored Procedure

AI Function Studio provides `SNOWFLAKE.CORTEX.EVALUATE_AI_FUNCTION()` — a stored procedure that:
1. Runs your function on every row in the test table
2. Compares predictions to ground truth using your chosen metric
3. Returns an aggregate score and stores per-row detail in a Snowflake Experiment

In [ ]:
-- Evaluate V2 with LLM Judge (best for multi-field VARIANT output)
CALL SNOWFLAKE.CORTEX.EVALUATE_AI_FUNCTION(
    'AI_STUDIO_DEMO.PUBLIC.CLASSIFY_SUPPORT_TICKET_V2',
    'AI_STUDIO_DEMO.PUBLIC.EVALUATION_TEST_DATA',
    ARRAY_CONSTRUCT('TICKET_TEXT'),
    'EXPECTED_OUTPUT',
    'llm_judge',
    'claude-sonnet-4-6',
    NULL,
    NULL,
    PARSE_JSON('{"task_description": "Classify enterprise B2B support tickets. Output must correctly identify division, issue_type, priority, customer_segment, and sla_flag."}'),
    500,
    NULL,
    NULL
);

---
## Evaluation Complete

We now have quantified accuracy for both V1 and V2. The evaluation framework enables:
- **Rapid iteration** — change the prompt, re-evaluate, compare
- **Custom metrics** — weight fields by business importance
- **Formal tracking** — stored procedure creates Snowflake Experiments for audit trails

Proceed to **Notebook 4: Model Comparison & RBAC** to compare multiple LLMs and secure the function.

In [ ]:
-- Mark this notebook as completed
MERGE INTO DEMO_STATE AS t
USING (SELECT 'evaluation' AS NOTEBOOK_ID, CURRENT_TIMESTAMP() AS COMPLETED_AT) AS s
ON t.NOTEBOOK_ID = s.NOTEBOOK_ID
WHEN MATCHED THEN UPDATE SET COMPLETED_AT = s.COMPLETED_AT
WHEN NOT MATCHED THEN INSERT (NOTEBOOK_ID, COMPLETED_AT) VALUES (s.NOTEBOOK_ID, s.COMPLETED_AT);